In [ ]:
import sys
import os
from os.path import *
sys.path.append(abspath(join(dirname(os.getcwd()))))

import rasterio

from utils import fast_imshow

In [ ]:
prob_map_path = "/home/luizluz/Documentos/multi-task-fcn/bioflore_data/v01/iter_001/region_0/raster_prediction/join_prob_0.8.TIF"

depth_map_path = "/home/luizluz/Documentos/multi-task-fcn/bioflore_data/v01/iter_001/region_0/raster_prediction/depth_0.8.TIF"

In [ ]:

# Read the probability map
with rasterio.open(prob_map_path) as prob_src:
    prob_map = prob_src.read(1)  # Read first band

# Read the depth map
with rasterio.open(depth_map_path) as depth_src:
    depth_map = depth_src.read(1)  # Read first band


In [ ]:
depth_map.max()

In [ ]:
import cv2
import numpy as np

depth_thr = 0.7
prob_thr = 0.5
sigma = 30

# Converter sigma para tamanho de kernel (ímpar)
# Regra prática: kernel_size ≈ 6*sigma
ksize = int(6 * sigma) | 1  # Garante número ímpar

# Aplicar suavização Gaussiana com OpenCV (muito mais rápido)
depth_gauss = cv2.GaussianBlur(depth_map, (ksize, ksize), sigma)
prob_gauss = cv2.GaussianBlur(prob_map, (ksize, ksize), sigma)

# Filtro combinado
mask = ((depth_gauss.astype("uint16") + prob_gauss.astype("uint16")) > ((depth_thr+ prob_thr)*255))
# Ou seja: (depth_gauss + prob_gauss) > 1.2


In [ ]:
(depth_gauss + prob_gauss).dtype

In [ ]:
depth_gauss.max()

In [ ]:
prob_gauss.max()

In [ ]:
confidence_map = (depth_gauss.astype("uint16") + prob_gauss.astype("uint16"))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.filters import threshold_otsu

# Amostragem aleatória de pixels do confidence_map
sample_size = 50000
confidence_sample = confidence_map.flatten()
if confidence_sample.size > sample_size:
    confidence_sample = np.random.choice(confidence_sample, size=sample_size, replace=False)
confidence_sample = confidence_sample / 255  # normaliza para [0, 1] sempre

# Calcular Otsu na amostra normalizada
confidence_otsu = threshold_otsu(confidence_sample)

plt.figure(figsize=(10, 5))
sns.histplot(confidence_sample, bins=100, kde=True)
plt.axvline(confidence_otsu, color="red", linestyle="--", label=f"Otsu = {confidence_otsu:.3f}")
plt.title("Distribuição dos pixels do confidence_map (amostra)")
plt.xlabel("Valor do pixel (confidence, normalizado)")
plt.ylabel("Contagem")
plt.legend()
plt.show()

In [ ]:
from skimage.filters import threshold_otsu

# Plotando o histograma para depth_gauss
plt.figure(figsize=(10, 5))
depth_sample = depth_gauss.flatten()
if depth_sample.size > sample_size:
    depth_sample = np.random.choice(depth_sample, size=sample_size, replace=False)
depth_sample_norm = depth_sample / 255  # normaliza para [0, 1]

# Calcule o threshold Otsu na amostra normalizada
depth_otsu = threshold_otsu(depth_sample_norm)
sns.histplot(depth_sample_norm, bins=100, kde=True)
plt.axvline(depth_otsu, color="red", linestyle="--", label=f"Otsu = {depth_otsu:.3f}")
plt.title("Distribuição dos pixels do depth_gauss (amostra)")
plt.xlabel("Valor do pixel (depth, normalizado)")
plt.ylabel("Contagem")
plt.legend()
plt.show()

# Plotando o histograma para prob_gauss
plt.figure(figsize=(10, 5))
prob_sample = prob_gauss.flatten()
if prob_sample.size > sample_size:
    prob_sample = np.random.choice(prob_sample, size=sample_size, replace=False)
prob_sample_norm = prob_sample / 255  # normaliza para [0, 1]

# Calcule o threshold Otsu na amostra normalizada
prob_otsu = threshold_otsu(prob_sample_norm)
sns.histplot(prob_sample_norm, bins=100, kde=True)
plt.axvline(prob_otsu, color="red", linestyle="--", label=f"Otsu = {prob_otsu:.3f}")
plt.title("Distribuição dos pixels do prob_gauss (amostra)")
plt.xlabel("Valor do pixel (probabilidade, normalizado)")
plt.ylabel("Contagem")
plt.legend()
plt.show()


In [ ]:
from skimage.filters import threshold_otsu

threshold_value = threshold_otsu(confidence_map)

threshold_value/255

In [ ]:
fast_imshow(mask)

In [ ]:
prob_otsu

In [ ]:
fast_imshow(prob_map > (prob_otsu*255))

In [ ]:
fast_imshow(depth_map > (depth_otsu*255))

In [ ]:
fast_imshow(confidence_map > 255)